In [ ]:
import json
import subprocess
import os
import re

# ── 1. Get existing resources ──────────────────────────────────────────────
resources = json.loads(subprocess.run(
    ["wb", "resource", "list", "--format=json"],
    capture_output=True, text=True, check=True
).stdout)

existing_ids = {r["id"] for r in resources}

# ── 2. Create buckets if they don't already exist ──────────────────────────
for name, desc in [
    ("workspace-input",   "Input files"),
    ("workspace-output",  "Output files"),
    ("workspace-scripts", "Script files"),
    ("workspace-GenDat", "Genetic data files"),
]:
    if name not in existing_ids:
        print(f"Creating {name}...")
        subprocess.run([
            "wb", "resource", "create", "gcs-bucket",
            f"--name={name}",
            "--cloning=COPY_NOTHING",
            f"--description={desc}"
        ], check=True)
        print(f"✅ Created {name}")
    else:
        print(f"⏭️  {name} already exists, skipping")

# ── 3. Re-fetch resources after creation ──────────────────────────────────
resources = json.loads(subprocess.run(
    ["wb", "resource", "list", "--format=json"],
    capture_output=True, text=True, check=True
).stdout)

# ── 4. Extract bucket paths and CDR ───────────────────────────────────────
bucket_input = bucket_output = bucket_scripts = dataset = None

for r in resources:
    if r["resourceType"] == "GCS_BUCKET":
        if r["id"] == "workspace-input":
            bucket_input   = f"gs://{r['bucketName']}"
        elif r["id"] == "workspace-output":
            bucket_output  = f"gs://{r['bucketName']}"
        elif r["id"] == "workspace-scripts":
            bucket_scripts = f"gs://{r['bucketName']}"
        elif r["id"] == "workspace-GenDat":
            bucket_GenDat = f"gs://{r['bucketName']}"
    elif r["resourceType"] in ["BQ_DATASET", "BIGQUERY_DATASET"]:
        if dataset is None and re.match(r"^C\d{4}Q\d+R\d+$", r["datasetId"]):
            dataset = f"{r['projectId']}.{r['datasetId']}"

# ── 5. Save to ~/.bashrc (deduplicated) ───────────────────────────────────
vars_to_save = {
    "WORKSPACE_BUCKET":         bucket_input   or "NOT_FOUND",
    "WORKSPACE_BUCKET_OUTPUT":  bucket_output  or "NOT_FOUND",
    "WORKSPACE_BUCKET_SCRIPTS": bucket_scripts or "NOT_FOUND",
    "WORKSPACE_BUCKET_GENDAT":  bucket_GenDat  or "NOT_FOUND",
    "WORKSPACE_CDR":            dataset        or "NOT_FOUND",
}

bashrc_path = os.path.expanduser("~/.bashrc")
if os.path.exists(bashrc_path):
    with open(bashrc_path, "r") as f:
        lines = [l for l in f.readlines()
                 if not any(v in l for v in list(vars_to_save.keys()) + ["# Verily Workbench variables"])]
else:
    lines = []

lines.append("\n# Verily Workbench variables\n")
for var, val in vars_to_save.items():
    lines.append(f'export {var}="{val}"\n')

with open(bashrc_path, "w") as f:
    f.writelines(lines)

print("\nVariables saved:")
for var, val in vars_to_save.items():
    print(f"  {var}: {val}")
print("\n✅ Setup complete!")